# Initial Designs

In this notebook, I am pasting some old or initial designs I have worked one prior to settling with the final architecture.

**These are not functional code blocks!!**

## Initial plot_image

In [ ]:
def plot_image(img1_address, img3_address, img2_address, num_rows, row_index):
    image1 = cv2.imread(img1_address)
    image1 = cv2.cvtColor(image1.astype("uint8"), cv2.COLOR_BGR2RGB)
    image2 = cv2.imread(img2_address)
    image2 = cv2.cvtColor(image2.astype("uint8"), cv2.COLOR_BGR2RGB)
    image3 = cv2.imread(img3_address)
    image3 = cv2.cvtColor(image3.astype("uint8"), cv2.COLOR_BGR2RGB)

    plt.subplot(num_rows, 3, 3 * row_index + 1)
    plt.title("Image 1 (Input)", fontdict={"fontsize": 10})
    plt.axis("off")
    plt.imshow(image1)

    plt.subplot(num_rows, 3, 3 * row_index + 2)
    plt.title("Image 3 (Input)", fontdict={"fontsize": 10})
    plt.axis("off")
    plt.imshow(image3)

    plt.subplot(num_rows, 3, 3 * row_index + 3)
    plt.title("Image 2 (In-Between)", fontdict={"fontsize": 10})
    plt.axis("off")
    plt.imshow(image2)


num_rows = 4

plt.figure(figsize=(10, 8))
for idx in range(num_rows):
    img1_address = train_df["Image 1 (Input)"][idx]
    img2_address = train_df["Image 2 (Output)"][idx]
    img3_address = train_df["Image 3 (Input)"][idx]
    plot_image(img1_address, img3_address, img2_address, num_rows, idx)
plt.show()

## Initial Dataset Design

In [ ]:
class FrameInterpolation(Dataset):
    def __init__(self, df, augment=False):
        super().__init__()
        # Storing the lists of paths, not the actual images
        self.img1_paths = df["Image 1 (Input)"].values
        self.img3_paths = df["Image 3 (Input)"].values
        self.img2_paths = df["Image 2 (Output)"].values

        self.augment = augment 
        

    def __getitem__(self, idx):
        path1 = self.img1_paths[idx]
        path3 = self.img3_paths[idx]
        path2 = self.img2_paths[idx]

        img1 = cv2.imread(path1)
        img3 = cv2.imread(path3)
        img2 = cv2.imread(path2)

        # Convert BGR to RGB
        img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
        img3 = cv2.cvtColor(img3, cv2.COLOR_BGR2RGB)
        img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

        # Normalise 0-1
        img1 = (img1 / 255.0).astype(np.float32)
        img3 = (img3 / 255.0).astype(np.float32)
        img2 = (img2 / 255.0).astype(np.float32)

        # To Tensor
        t_img1 = torch.from_numpy(img1).permute(2, 0, 1)
        t_img3 = torch.from_numpy(img3).permute(2, 0, 1)
        t_img2 = torch.from_numpy(img2).permute(2, 0, 1)

        # Data Augmentation (only runs if class "augment" is true)
        if self.augment:
            if random.random() > 0.5: # Random Horizontal Flip (50% chance)
                t_img1 = TF.hflip(t_img1)
                t_img3 = TF.hflip(t_img3)
                t_img2 = TF.hflip(t_img2)

            if random.random() > 0.5: # Random Rotation (50% chance)
                angle = random.uniform(-15, 15)
                t_img1 = TF.rotate(t_img1, angle)
                t_img3 = TF.rotate(t_img3, angle)
                t_img2 = TF.rotate(t_img2, angle)

            if random.random() > 0.5: # Random Brightness (50% chance)
                brightness = random.uniform(0.9, 1.1)
                t_img1 = TF.adjust_brightness(t_img1, brightness)
                t_img3 = TF.adjust_brightness(t_img3, brightness)
                t_img2 = TF.adjust_brightness(t_img2, brightness)
        
        # t_img1 = f.resize(t_img1, (128, 128)) # try more different data augmentation, rotation is -15 to +15 degrees only
        # t_img2 = f.resize(t_img2, (128, 128))
        # t_img3 = f.resize(t_img3, (128, 128))
        
        return (t_img1, t_img3), t_img2

    def __len__(self):
        return len(self.img1_paths)

## Initial Model Design

In [ ]:
class FrameInterpolator(nn.Module):
    # def __init__(self):
    #     super(FrameInterpolator, self).__init__()

    #     # Encoder
    #     self.enc1 = nn.Conv2d(6, 64, kernel_size=3, padding=1)
    #     self.enc2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
    #     self.enc3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
    #     self.pool = nn.MaxPool2d(2, stride=2)

    #     # Decoder
    #     self.dec1 = nn.Conv2d(256, 128)
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            # Layer 1
            nn.Conv2d(6, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Layer 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Layer 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        # Decoder
        self.decoder = nn.Sequential(
            # Layer 1
            nn.Upsample(scale_factor=2, mode="bilinear"),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            # Layer 2
            nn.Upsample(scale_factor=2, mode="bilinear"),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            # Layer 3
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
        )

    def forward(self, img1, img3):
        input = torch.cat((img1, img3), dim=1) # torch.add can also try torch.minus as additional input
        features = self.encoder(input)  # Run Encoder
        output = self.decoder(features)  # Run Decoder
        return output

## Initial Training Loop(s)

In [ ]:
best_score = 0.0

epochs = 15
total_time = time.time()
for epoch in range(epochs):
    start_time = time.time()

    print(f"Epoch: {epoch + 1}")
    loss, accuracy = mlutils.train(
        model_compiled, train_loader, optimizer, loss_function, accuracy_metric, device, train_N
    )
    print(f"Train Loss {loss:.04f} | Accuracy {accuracy:.4f}")
    train_loss.append(loss)
    train_accuracy.append(accuracy)

    # Validate
    loss, accuracy = mlutils.validate(model_compiled, valid_loader, loss_function, accuracy_metric, device, valid_N)
    print(f"Valid Loss {loss:.04f} | Accuracy {accuracy:.4f}")
    valid_loss.append(loss)
    valid_accuracy.append(accuracy)

    elapsed_time = time.time() - start_time
    print(f"Elapsed Time: {elapsed_time:.1f}s")

    if accuracy > best_score:
        best_score = accuracy
        torch.save(model_compiled.state_dict(), "best_frame_interpolator.pth")
        print("New Best!")

    print()

total_elapsed_time = time.time() - total_time
print(f"Training Complete. Total Time: {total_elapsed_time / 60:.2f} minutes")

In [ ]:
# Early stopping to circumvent overfitting
stop_limit = 10
stop_counter = 0

epochs = 183
total_time = time.time()
print("Training started.")
for epoch in range(start_epoch, epochs):
    start_time = time.time()

    # Train
    t_loss, t_accuracy = fiutils.train(
        model_compiled, train_loader, optimizer, loss_function, accuracy_metric, device, train_N
    )
    train_loss.append(t_loss)
    train_accuracy.append(t_accuracy)

    # Validate
    v_loss, v_accuracy = fiutils.validate(model_compiled, valid_loader, loss_function, accuracy_metric, device, valid_N)
    valid_loss.append(v_loss)
    valid_accuracy.append(v_accuracy)

    # Print
    if epoch%1 == 0: # Every (ideally) 5 epochs
        elapsed_time = time.time() - start_time
        print(f"Epoch: {epoch + 1}")
        print(f"Train Loss {t_loss:.04f} | Accuracy {t_accuracy:.4f}")
        print(f"Valid Loss {v_loss:.04f} | Accuracy {v_accuracy:.4f}")
        print(f"Elapsed Time: {elapsed_time:.1f}s")

    # Check session best
    if v_accuracy > session_best_score:
        session_best_score = v_accuracy
        torch.save(model_compiled.state_dict(), session_best_path)
        print("New Session Best!")
        stop_counter = 0 # Reset counter
    else:
        stop_counter += 1

    # Check global best
    if v_accuracy > global_best_score:
        global_best_score = v_accuracy
        torch.save(model_compiled.state_dict(), global_best_path)
        with open(score_file_path, "w") as f:
            f.write(str(global_best_score))
        print("New All-Time Best!")

    # Save checkpoint
    torch.save({
        "epoch": epoch,
        "model_state_dict": model_compiled.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_score": session_best_score,

        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
    }, checkpoint_path)

    # Early Stopping
    if stop_counter >= stop_limit:
        print("Model has stopped improving, training will end now.")
        break

    print()

total_elapsed_time = time.time() - total_time
print(f"Training Complete. Total Time: {total_elapsed_time / 60:.2f} minutes")

## Initial Result Visualisation

In [ ]:
random_test_loader = DataLoader(test_loader.dataset, batch_size=BATCH_SIZE, shuffle=True)

model_compiled.eval()
with torch.no_grad():
    (img1_display, img3_display), img2_display = next(iter(random_test_loader))
    output = model_compiled(img1_display.to(device), img3_display.to(device))

    num_samples = 4
    fig, axes = plt.subplots(nrows=num_samples * 2, ncols=3, figsize=(12, 4 * num_samples))  # 12, 4 * num_samples

    # plt.figure(figsize = (15, 12))
    for i in range(num_samples):
        row_real = i * 2
        row_output = i * 2 + 1

        # The Ground Truth
        # Input 1
        axes[row_real, 0].imshow(fiutils.display_image(img1_display[i]))
        axes[row_real, 0].set_title("Image 1 (Input)")
        axes[row_real, 0].axis("off")
        # Real Image
        axes[row_real, 1].imshow(fiutils.display_image(img2_display[i]))
        axes[row_real, 1].set_title("Image 2 (Ground Truth)")
        axes[row_real, 1].axis("off")
        # Input 3
        axes[row_real, 2].imshow(fiutils.display_image(img3_display[i]))
        axes[row_real, 2].set_title("Image 3 (Input)")
        axes[row_real, 2].axis("off")

        # The AI Output
        # Input 1
        axes[row_output, 0].imshow(fiutils.display_image(img1_display[i]))
        axes[row_output, 0].axis("off")
        # AI Image
        axes[row_output, 1].imshow(fiutils.display_image(output[i]))
        axes[row_output, 1].set_title("Image 2 (AI Output)")
        axes[row_output, 1].axis("off")
        # Input 3
        axes[row_output, 2].imshow(fiutils.display_image(img3_display[i]))
        axes[row_output, 2].axis("off")

    plt.show()